DATA ANALYSIS

Step 1: import the data

In [ ]:
# Import the data analysis library
import pandas as pd

# Import the data
table = pd.read_csv("customer_churn.csv")

Step 2: view the data

In [ ]:
# View the data
display(table)

Step 3: fix problems in the dataset
- data unnecessary for the analysis -> remove
- empty information -> remove or fill in manually
- information in the wrong format -> fix

In [ ]:
# remove columns unnecessary for the analysis
table = table.drop(columns="customer_id")
display(table)

In [ ]:
# view general information about the table
display(table.info())

In [ ]:
# remove rows with empty columns
table = table.dropna()
display(table.info())

Step 4: perform an initial analysis
- how many customers churned?

In [ ]:
#show how many customers churned
count = table["churn"].value_counts()
percentage = table["churn"].value_counts(normalize=True)

churn_summary = pd.DataFrame({
    "Count": count,
    "Percentage": percentage.map("{:.2%}".format)
})

churn_summary = churn_summary.rename(index={1.0: "Churned", 0.0: "Did not churn"})
churn_summary.index.name = None

display(churn_summary)

- Initial conclusion: 56.71% of customers churned

Step 5: analyze the main causes of customer churn
- generate charts
- display the charts

In [ ]:
import plotly.express as px

my_colors=["#FC1313","#3860FF"]

for column in table.columns:
    chart = px.histogram(table, 
                           x=column, 
                           color = "churn", 
                           color_discrete_sequence=my_colors, 
                           width=600)
    chart.show()

Identifying the causes:
- Everyone older than 50 churned
    - Analyze other factors to confirm whether age is actually a cause
- Everyone with more than 5 calls to the call center churned. At 5 calls, the vast majority had already churned.
    - Solve the problem by the 4th call.
- Everyone with a payment delay of more than 20 days churned.
    - Send a delay notification on the 10th day of delay.
- Everyone on a monthly plan churned
    - Offer attractive discounts for semiannual and annual plans

Step 6: analyze the impact of solving the observed problems

1. Calls to the call center
    - up to 4 calls

2. Payment delay
    - up to 20 days

3. Monthly plan
    - no monthly plan

In [ ]:
#original table with up to 4 calls
table_1 = table[table["support_calls"] < 5]

#original table with up to 20 days of payment delay
table_2 = table[table["payment_delay"] < 21]

#original table without a monthly contract
table_3 = table[table["contract_length"] != "Monthly"]

#comparative histograms
legend_colors = {"Churned": "#FC1313", "Did not churn": "#3860FF"}

def churn_comparison(df, title):
    count = df["churn"].value_counts()
    percentage = df["churn"].value_counts(normalize=True)

    summary = pd.DataFrame({"Count": count, "Percentage": percentage})
    summary.index.name = None
    summary["Percentage"] = summary["Percentage"].map("{:.2%}".format)

    print(title)
    display(summary)

churn_comparison(table_1, "Churn rate - Up to 4 calls to the call center")
churn_comparison(table_2, "Churn rate - Up to 20 days of payment delay")
churn_comparison(table_3, "Churn rate - No monthly plan")


def legend(tab):
    return tab["churn"].map({1.0: "Churned", 0.0: "Did not churn"})

hist_calls = px.histogram(table, 
                             x="support_calls", 
                             color=legend(table), 
                             color_discrete_map=legend_colors)
hist_table_1 = px.histogram(table_1, 
                             x="support_calls", 
                             color=legend(table_1), 
                             color_discrete_map=legend_colors)

hist_payment_delay = px.histogram(table, 
                                 x="payment_delay", 
                                 color=legend(table), 
                                 color_discrete_map=legend_colors)
hist_table_2 = px.histogram(table_2, 
                              x="payment_delay", 
                              color=legend(table_2), 
                              color_discrete_map=legend_colors)

hist_contract = px.histogram(table, 
                              x="contract_length", 
                              color=legend(table), 
                              color_discrete_map=legend_colors)
hist_table_3 = px.histogram(table_3, 
                              x="contract_length", 
                              color=legend(table_3), 
                              color_discrete_map=legend_colors)


from plotly.subplots import make_subplots
combined_charts = make_subplots(rows=3, cols=2,
                    subplot_titles=("Original", "Up to 4 calls",
                                    "Original", "Up to 20 days of delay",
                                    "Original", "No monthly plan"))

positions = [
    (hist_calls,         1, 1),
    (hist_table_1,       1, 2),
    (hist_payment_delay, 2, 1),
    (hist_table_2,       2, 2),
    (hist_contract,      3, 1),
    (hist_table_3,       3, 2),
]

for fig, row, col in positions:
    for trace in fig.data:
        # show the legend only the first time each color appears
        trace.showlegend = (row == 1 and col == 1)
        combined_charts.add_trace(trace, row=row, col=col)

combined_charts.update_layout(height=900, barmode="overlay", legend_title_text="Churn")
combined_charts.show()